In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [2]:
df_ai = pd.read_csv("../data/Raw_data/train.csv", nrows = 15000)

df_ai.head()

,prompt,Human_story,gemma-2-9b,mistral-7B,qwen-2-72B,llama-8B,accounts/yi-01-ai/models/yi-large,GPT_4-o
0,The Human Toll of Nuclear Testing,Comments\nThe U.S. bombings thatended World Wa...,## The Unseen Scars: The Enduring Human Toll o...,\n\nTitle: The Atomic Aftermath: The U.S. Bomb...,Human Toll of Nuclear Testing: Unraveling the ...,The Ghostly Whispers of Trinity The Enduring ...,The human toll of nuclear testing is a profoun...,# The Human Toll of Nuclear Testing: A Legacy ...
1,"In the age of coronavirus, the only way you ca...",new video loaded:Messages From Quarantine\ntra...,"## In the Age of Coronavirus, the Only Way You...","Title: ""Exploring Milan in the Age of Coronav...","In the Age of Coronavirus, the Only Way to See...",**The New York Times**\n\n**IN THE AGE OF CORO...,**Title: Navigating Milan in the Age of Corona...,"**Title: In the Age of Coronavirus, the Only W..."
2,"Roberta Karmel, First Woman Named to the S.E.C...","Supported by\nRoberta Karmel, First Woman Name...","## Roberta Karmel, First Woman Named to the S....",\n\nTitle: A Commissioner's Crusade: From SEC ...,"Karmel, a trailblazing lawyer and academic who...",Roberta Karmel Trailblazing Force at the SEC ...,I'm sorry to hear that Roberta Karmel has pass...,"**Roberta Karmel, First Woman Named to the S.E..."
3,"Summer Reading Contest, Week 2: What Got Your ...",Supported by\nContests\nSummer Reading Contest...,## Summer Reading Contest: Week 2 - What Caug...,\n\nTitle: NYT Invites Readers to Share Their ...,"er Reading Contest, Week 2: What Got Your Atte...",What Caught Your Eye The New York Times Summe...,"I'm sorry, but I can't access or provide real-...","**Summer Reading Contest, Week 2: What Got You..."
4,Photos posted this week on @nytimes took our f...,The Week on Instagram\n@heislerphoto was one o...,## From Granite State to Lakeside Paradise: Ne...,Title: Mike Eckhaus and Zoe Latta Unveil Stun...,Title: Mike Eckhaus and Zoe Latta Unveil Their...,**MIKE ECKHAUS AND ZOE LATTA UNVEIL THEIR LATE...,**Mike Eckhaus and Zoe Latta Unveil Their Late...,Error: Error communicating with OpenAI: HTTPSC...


In [3]:
df_ai = df_ai[["GPT_4-o"]].copy()
df_ai["generated"] = 1
df_ai = df_ai.rename(columns={"GPT_4-o":"text"})
df_ai.tail()

,text,generated
7316,**Title: Discovering Dumpling Delights: Unveil...,1
7317,Error: Error communicating with OpenAI: HTTPSC...,1
7318,Error: Error communicating with OpenAI: HTTPSC...,1
7319,"**Is It Storage or Art? If It’s Hard to Tell, ...",1
7320,**Title: Evacuation Concludes in Aleppo as Ass...,1


In [4]:
# Checking length of df_ai

df_ai["len"] = df_ai["text"].str.split().apply(len)

print("AI length stats:")
print(df_ai["len"].describe())

AI length stats:
count    7321.000000
mean      570.005874
std       134.871287
min        11.000000
25%       532.000000
50%       585.000000
75%       638.000000
max      2099.000000
Name: len, dtype: float64


In [5]:
MIN_LEN = df_ai["len"].quantile(0.1)   # 10th percentile
MAX_LEN = df_ai["len"].quantile(0.9)   # 90th percentile

MIN_LEN = int(MIN_LEN)
MAX_LEN = int(MAX_LEN)

print("Target length range:", MIN_LEN, "-", MAX_LEN)

Target length range: 483 - 689


In [6]:
# filter rows based on text length

df_ai_0 = df_ai.copy()

# recompute word length 
df_ai_0["len"] = df_ai_0["text"].astype(str).str.split().apply(len)

# keep only rows with at least MIN_LEN words
df_ai_0 = df_ai_0[df_ai_0["len"] >= MIN_LEN]

# cut text to MAX_LEN words
df_ai_0["text"] = (
    df_ai_0["text"]
    .astype(str)
    .str.split()
    .str[:MAX_LEN]
    .str.join(" ")
)

# drop helper column
df_ai_0 = df_ai_0.drop(columns=["len"]).reset_index(drop=True)

In [7]:
df_ai.info()

<class 'pandas.DataFrame'>
RangeIndex: 7321 entries, 0 to 7320
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   text       7321 non-null   str  
 1   generated  7321 non-null   int64
 2   len        7321 non-null   int64
dtypes: int64(2), str(1)
memory usage: 27.4 MB


In [8]:
df_ai_0.info()

<class 'pandas.DataFrame'>
RangeIndex: 6590 entries, 0 to 6589
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       6590 non-null   object
 1   generated  6590 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 103.1+ KB


In [9]:
df_ai_0["text"] = df_ai_0["text"].astype("string")

In [10]:
print(df_ai_0.head(5).to_string())


In [11]:
# df_ai_1 = pd.read_csv("../data/Separated_csv_1/responses_csv/responses_processed.csv")

In [12]:
# Checking length of df_ai

# df_ai_1["len"] = df_ai_1["text"].str.split().apply(len)

# print("AI length stats:")
# print(df_ai_1["len"].describe())


In [18]:
MIN_LEN = 283

# paths & setup
base_path = "../data/Separated_csv_1/non_native_human/long_text_chunks/"
files = [f"{base_path}long_text_chunk_{i}.csv" for i in range(10)]

dfs = []

for file in files:
    df = pd.read_csv(file)

    # make sure text is string
    df["text"] = df["text"].astype(str)

    # compute word length (SAME logic as AI)
    df["len"] = df["text"].str.split().apply(len)

    # keep rows with at least MIN_LEN words
    df = df[df["len"] >= MIN_LEN]

    # cut text to MAX_LEN words
    df["text"] = (
        df["text"]
        .str.split()
        .str[:MAX_LEN]
        .str.join(" ")
    )

    # drop helper column
    df = df.drop(columns=["len"])

    dfs.append(df)

# merge all human chunks
df_human_0 = pd.concat(dfs, ignore_index=True)

# save combined CSV
output_path = "../data/Separated_csv_1/non_native_human/df_human_0.csv"
df_human_0.to_csv(output_path, index=False)

print("Final human dataframe shape:", df_human_0.shape)
print("Saved to:", output_path)

Final human dataframe shape: (352, 2)
Saved to: ../data/Separated_csv_1/non_native_human/df_human_0.csv


In [19]:
df_human_0.info()

<class 'pandas.DataFrame'>
RangeIndex: 352 entries, 0 to 351
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       352 non-null    object
 1   generated  352 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 5.6+ KB


In [23]:
print(df_human_0.head(10).to_string())

In [24]:
import re
from collections import Counter

def is_repetitive(text, word_ratio_thresh=0.35, unique_ratio_thresh=0.2):
    """
    Returns True if text is overly repetitive
    """
    # tokenize words (lowercase)
    words = re.findall(r"\b\w+\b", text.lower())

    if len(words) == 0:
        return True

    counts = Counter(words)

    # 1) dominant word ratio
    most_common_word, freq = counts.most_common(1)[0]
    if freq / len(words) >= word_ratio_thresh:
        return True

    # 2) vocabulary diversity
    if len(counts) / len(words) <= unique_ratio_thresh:
        return True

    # 3) repeated symbols (e.g., !!!!! ---- ====)
    if re.search(r"(.)\1{5,}", text):
        return True

    return False


In [25]:
df_human_0["is_repetitive"] = df_human_0["text"].apply(is_repetitive)

# keep only clean rows
df_human_0 = df_human_0[~df_human_0["is_repetitive"]]

# cleanup
df_human_0 = df_human_0.drop(columns=["is_repetitive"]).reset_index(drop=True)

print("After repetition filtering:", df_human_0.shape)


After repetition filtering: (207, 2)


In [26]:
df_ai_0["is_repetitive"] = df_ai_0["text"].apply(is_repetitive)
df_ai_0 = df_ai_0[~df_ai_0["is_repetitive"]]
df_ai_0 = df_ai_0.drop(columns=["is_repetitive"]).reset_index(drop=True)


In [27]:
print(df_human_0.head(10).to_string())

In [29]:
df_ai_0.info()

<class 'pandas.DataFrame'>
RangeIndex: 6563 entries, 0 to 6562
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       6563 non-null   string
 1   generated  6563 non-null   int64 
dtypes: int64(1), string(1)
memory usage: 25.5 MB


In [30]:
df_human_1 = pd.read_csv("../data/Raw_data/ielts_writing_dataset.csv")

df_human_1.info()

<class 'pandas.DataFrame'>
RangeIndex: 1435 entries, 0 to 1434
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Task_Type           1435 non-null   int64  
 1   Question            1435 non-null   str    
 2   Essay               1435 non-null   str    
 3   Examiner_Commen     62 non-null     str    
 4   Task_Response       0 non-null      float64
 5   Coherence_Cohesion  0 non-null      float64
 6   Lexical_Resource    0 non-null      float64
 7   Range_Accuracy      0 non-null      float64
 8   Overall             1435 non-null   float64
dtypes: float64(5), int64(1), str(3)
memory usage: 2.5 MB


In [31]:
df_human_1 = df_human_1[["Essay"]]

# rename to match your pipeline 
df_human_1 = df_human_1.rename(columns={"Essay": "text"})

# add generated column with 0
df_human_1["generated"] = 0

print(df_human_1.head())
print(df_human_1.shape)

                                                text  generated
0  Between 1995 and 2010, a study was conducted r...          0
1  Poverty represents a worldwide crisis. It is t...          0
2  The left chart shows the population change hap...          0
3  Human beings are facing many challenges nowada...          0
4  Information about the thousands of visits from...          0
(1435, 2)


In [33]:
# Checking length of df_human_1

df_human_1["len"] = df_human_1["text"].str.split().apply(len)

print("Human length stats:")
print(df_human_1["len"].describe())

print(f"10% = {df_human_1["len"].quantile(0.1)}")
print(f"90% = {df_human_1["len"].quantile(0.9)}")


Human length stats:
count    1435.000000
mean      256.732404
std        81.324270
min       116.000000
25%       181.000000
50%       259.000000
75%       307.000000
max       577.000000
Name: len, dtype: float64
10% = 159.0
90% = 365.60000000000014


In [34]:
df_human = pd.concat(
    [df_human_0, df_human_1],
    ignore_index=True
)

print(df_human["generated"].value_counts())

generated
0    1642
Name: count, dtype: int64


In [35]:
df_human.to_csv("temp.csv", index=False)


In [3]:
df_1 = pd.read_csv('temp.csv')

In [5]:
df_1.info()

<class 'pandas.DataFrame'>
RangeIndex: 1642 entries, 0 to 1641
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   text       1642 non-null   str    
 1   generated  1642 non-null   int64  
 2   len        1435 non-null   float64
dtypes: float64(1), int64(1), str(1)
memory usage: 2.5 MB


In [7]:
df_2 = pd.read_csv('../data/Separated_csv_1/filtered_output_human.csv')

In [8]:
df_2.info()

<class 'pandas.DataFrame'>
RangeIndex: 2499 entries, 0 to 2498
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   text       2499 non-null   str  
 1   generated  2499 non-null   int64
dtypes: int64(1), str(1)
memory usage: 3.4 MB


In [ ]:
# Remove the 'len' column from df_1
df_1 = df_1.drop('len', axis=1)

# Combine df_1 and df_2
combined_df = pd.concat([df_1, df_2], ignore_index=True)

print(f"Total rows in combined dataframe: {len(combined_df)}")
print(combined_df.head())

Total rows in combined dataframe: 4141
                                                text  generated
0  i have avenged FW and i will continue to aveng...          0
1  shit , piss , fuck , cunt , cocksucker , mothe...          0
2  i have avenged FW and i will continue to aveng...          0
3  no not really l was actually flat out mapping ...          0
4  * `` but what , i asked , if , insisting on th...          0


In [10]:
combined_df.to_csv('../data/Separated_csv_1/combined_human_4141.csv', index=False)

In [13]:
df = pd.read_csv('../data/Raw_data/AI_human.csv')

# Filter rows where generated == 0.0 and take first 20,000
df_result = df[df['generated'] == 0.0].head(20000).reset_index(drop=True)

print(f"Rows taken: {len(df_result)}")
print(df_result.head())

Rows taken: 20000
                                                text  generated
0  Cars. Cars have been around since they became ...        0.0
1  Transportation is a large necessity in most co...        0.0
2  "America's love affair with it's vehicles seem...        0.0
3  How often do you ride in a car? Do you drive a...        0.0
4  Cars are a wonderful thing. They are perhaps o...        0.0


In [14]:
df_result.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   text       20000 non-null  str    
 1   generated  20000 non-null  float64
dtypes: float64(1), str(1)
memory usage: 46.4 MB


In [16]:
MIN_LEN = 200

df_3 = df_result[
    df_result["text"].str.split().apply(len) >= MIN_LEN
]

In [19]:
df_3.info()

<class 'pandas.DataFrame'>
Index: 18679 entries, 0 to 19999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   text       18679 non-null  str  
 1   generated  18679 non-null  int64
dtypes: int64(1), str(1)
memory usage: 45.3 MB


In [18]:
df_3["generated"] = df_3["generated"].astype(int)

In [20]:
# Combine combined_df and df_3
combined_df = pd.concat([combined_df, df_3], ignore_index=True)

print(f"Total rows in combined dataframe: {len(combined_df)}")
print(combined_df.head())

Total rows in combined dataframe: 22820
                                                text  generated
0  i have avenged FW and i will continue to aveng...          0
1  shit , piss , fuck , cunt , cocksucker , mothe...          0
2  i have avenged FW and i will continue to aveng...          0
3  no not really l was actually flat out mapping ...          0
4  * `` but what , i asked , if , insisting on th...          0


In [22]:
print("Human (0):", (combined_df["generated"] == 0).sum())
print("AI (1):", (combined_df["generated"] == 1).sum())


Human (0): 22820
AI (1): 0


In [23]:
combined_df.to_csv('../data/Separated_csv_1/combined_human_22820.csv', index=False)

In [2]:
df_ai_1 = pd.read_csv('../data/Separated_csv_1/responses_csv/responses_processed.csv')

In [3]:
df_ai_1.head()

,text,generated
0,"I appreciate your query, and as Grok, an AI bu...",1
1,Below is a comprehensive report on the topic o...,1
2,Below is a 1000-word monologue written in the ...,1
3,Below is a comprehensive SEO-optimized article...,1
4,"**Prompt:** \nIn a bustling modern city, towe...",1


In [6]:
df_ai_1 = df_ai_1.dropna(subset=["text"])
df_ai_1 = df_ai_1[df_ai_1["text"].str.strip() != ""]


In [7]:
df_ai_1.info()

<class 'pandas.DataFrame'>
Index: 158764 entries, 0 to 158999
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   text       158764 non-null  str  
 1   generated  158764 non-null  int64
dtypes: int64(1), str(1)
memory usage: 618.1 MB


In [8]:
# define limits
MIN_LEN = 200
CUT_LEN = 300

df_ai_1 = df_ai_1.copy()

# compute word length
df_ai_1["len"] = df_ai_1["text"].astype(str).str.split().apply(len)

# keep rows with at least 200 words
df_ai_1 = df_ai_1[df_ai_1["len"] >= MIN_LEN]

# cut text to max 300 words
df_ai_1["text"] = (
    df_ai_1["text"]
    .astype(str)
    .str.split()
    .str[:CUT_LEN]
    .str.join(" ")
)

# cleanup
df_ai_1 = df_ai_1.drop(columns=["len"]).reset_index(drop=True)

print(df_ai_1.shape)


(131836, 2)


In [14]:
df_ai_1_10k = df_ai_1.sample(n=14_700, random_state=42).reset_index(drop=True)


In [15]:
df_ai_1_10k.info()

<class 'pandas.DataFrame'>
RangeIndex: 14700 entries, 0 to 14699
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       14700 non-null  object
 1   generated  14700 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 229.8+ KB


In [16]:
df_ai_1_10k.to_csv('../data/Separated_csv_1/grok_ai_14700.csv', index=False)

In [12]:
df_ai = pd.read_csv("../data/Raw_data/train.csv")

In [13]:
df_ai.info()

<class 'pandas.DataFrame'>
RangeIndex: 7321 entries, 0 to 7320
Data columns (total 8 columns):
 #   Column                             Non-Null Count  Dtype
---  ------                             --------------  -----
 0   prompt                             7321 non-null   str  
 1   Human_story                        7295 non-null   str  
 2   gemma-2-9b                         7310 non-null   str  
 3   mistral-7B                         7316 non-null   str  
 4   qwen-2-72B                         7314 non-null   str  
 5   llama-8B                           7306 non-null   str  
 6   accounts/yi-01-ai/models/yi-large  7319 non-null   str  
 7   GPT_4-o                            7321 non-null   str  
dtypes: str(8)
memory usage: 153.2 MB


In [17]:
df_ai_1 = df_ai[['GPT_4-o']].copy()

df_ai_1.rename(columns={'GPT_4-o': 'text'}, inplace=True)

df_ai_1['generated'] = 1

In [18]:
# define limits
MIN_LEN = 200
CUT_LEN = 300

df_ai_1 = df_ai_1.copy()

# compute word length
df_ai_1["len"] = df_ai_1["text"].astype(str).str.split().apply(len)

# keep rows with at least 200 words
df_ai_1 = df_ai_1[df_ai_1["len"] >= MIN_LEN]

# cut text to max 300 words
df_ai_1["text"] = (
    df_ai_1["text"]
    .astype(str)
    .str.split()
    .str[:CUT_LEN]
    .str.join(" ")
)

# cleanup
df_ai_1 = df_ai_1.drop(columns=["len"]).reset_index(drop=True)

print(df_ai_1.shape)

(7032, 2)


In [19]:
df_ai_1.to_csv('../data/Separated_csv_1/gpt4o_ai_7032.csv', index=False)

In [20]:
for df in [df_ai_1, df_ai_1_10k]:
    df['text'] = df['text'].astype(str)
    df['generated'] = df['generated'].astype(int)


In [21]:
df_ai_combined = pd.concat(
    [df_ai_1, df_ai_1_10k],
    axis=0,
    ignore_index=True
)


In [22]:
df_ai_combined.dtypes
df_ai_combined['generated'].value_counts()


generated
1    21732
Name: count, dtype: int64

In [23]:
df_ai_combined.to_csv('../data/Separated_csv_1/combined_ai_21732.csv', index=False)

In [24]:
df_human_combined = pd.read_csv('../data/Separated_csv_1/combined_human_22820.csv')

In [25]:
df_combined = pd.concat(
    [df_ai_combined, df_human_combined],
    axis=0,
    ignore_index=True
)

In [26]:
df_combined.to_csv('../data/mixed_chunk_data/combined_ai_human.csv', index=False)
